# F-003-3e: YOLOv8 pico 精度改善モデル再学習 Notebook

精度改善パラメータ（#101）を適用した pico モデルを RGB 3ch で再学習し、
実機（EK-RA8P1）で人物検出が動作する状態にする。

## Issue #126 対応方針
- **方針A**: RGB 3ch で精度改善 pico モデルを再学習（推奨）
- MCU前処理 (`image_rgb565_to_rgb_int8`) が 3ch RGB 出力固定のため、3ch で学習
- pico モデル構成: depth=0.33, width=0.08, max_channels=256（動作実績あり）
- 精度改善パラメータ: epochs=300, lrf=0.001, cos_lr=True (#101)

## YOLOv8n からの変更点
- **width_multiple**: 0.25 -> 0.08 (pico, max_channels=256)
- **入力チャネル**: 3 (RGB)
- **クラス数**: 1 (person)

## 前提条件
- Google Colab (GPU ランタイム: T4 推奨)
- Google Drive に `fall_detection_dataset.zip` をアップロード済み

## ワークフロー
1. GPU確認・環境構築
2. データセット準備
3. カスタムモデルYAML作成
4. モデル学習 (精度改善パラメータ適用)
5. 精度評価 (mAP)
6. ONNX エクスポート
7. TFLite FP32/INT8 変換
8. モデルサイズ検証
9. 成果物ダウンロード

---
## Step 1: GPU確認・環境構築

**重要:** メニューの「ランタイム > ランタイムのタイプを変更」で **GPU (T4)** を選択してください。

### 初回実行時の注意
pip installセル実行後、**ランタイムを再起動**してからStep 1を再度実行してください。
Colabにプリインストールされたnumpyとの互換性問題を回避するためです。
（2回目以降はインストール済みなのですぐ完了します）

In [ ]:
# GPU 確認
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
    print('=== GPU が利用可能です ===')
else:
    print('WARNING: GPU が検出されません。ランタイムを GPU に変更してください。')

In [ ]:
# numpy互換性問題を回避してからインストール
!pip install -q numpy==1.26.4
!pip install -q ultralytics onnx onnx2tf onnxsim
print('=== インストール完了 ===')

---
## Step 2: データセット準備

### 事前準備 (ローカルPCで実行)

```bash
cd mimamori-sense/dataset/merged
zip -r fall_detection_dataset.zip images/ labels/
```

作成した `fall_detection_dataset.zip` を Google Drive のマイドライブ直下にアップロードしてください。

### 入力チャネルについて

このノートブックは `ch: 3` (RGB) で学習します。
MCU前処理 (`image_rgb565_to_rgb_int8`) が RGB 3ch 出力固定のため、
学習時と推論時の入力チャネルを一致させます。

In [ ]:
# Google Drive マウント
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

WORK_DIR = '/content/yolo_pico_train'
DATASET_DIR = os.path.join(WORK_DIR, 'dataset')
DATASET_GRAY_DIR = os.path.join(WORK_DIR, 'dataset_gray')
DATASET_ZIP = '/content/drive/MyDrive/fall_detection_dataset.zip'

os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)
print(f'作業ディレクトリ: {WORK_DIR}')

# データセット展開
if not os.path.isdir(DATASET_DIR):
    if os.path.isfile(DATASET_ZIP):
        print('データセット展開中...')
        !mkdir -p {DATASET_DIR} && unzip -q {DATASET_ZIP} -d {DATASET_DIR}
        print('展開完了')
    else:
        print(f'ERROR: {DATASET_ZIP} が見つかりません')
        print('Google Drive にアップロードしてください')
else:
    print('データセットは展開済みです')

# 検証
for split in ['train', 'val', 'test']:
    img_dir = os.path.join(DATASET_DIR, 'images', split)
    lbl_dir = os.path.join(DATASET_DIR, 'labels', split)
    if os.path.isdir(img_dir):
        img_count = len([f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.png', '.jpeg'))])
        lbl_count = len([f for f in os.listdir(lbl_dir) if f.endswith('.txt')]) if os.path.isdir(lbl_dir) else 0
        print(f'  {split}: images={img_count}, labels={lbl_count}')
    else:
        print(f'  WARNING: {img_dir} が見つかりません')

In [ ]:
# RGB 3ch で学習するため、Grayscale変換はスキップ
# データセットをそのまま使用する
print('=== RGB データセット使用 (Grayscale変換スキップ) ===')
print('MCU前処理 (image_rgb565_to_rgb_int8) が 3ch RGB 出力固定のため、')
print('RGB データセットで学習します。')
print()

# データセットの確認
for split in ['train', 'val', 'test']:
    img_dir = os.path.join(DATASET_DIR, 'images', split)
    if os.path.isdir(img_dir):
        img_count = len([f for f in os.listdir(img_dir)
                         if f.lower().endswith(('.jpg', '.png', '.jpeg'))])
        print(f'  {split}: images={img_count}')

In [ ]:
# data.yaml 作成 (RGB データセット)
data_yaml = f"""path: {DATASET_DIR}
train: images/train
val: images/val
test: images/test

nc: 1
names:
  0: person
"""

data_yaml_path = os.path.join(WORK_DIR, 'data.yaml')
with open(data_yaml_path, 'w') as f:
    f.write(data_yaml)

print(f'data.yaml 作成完了: {data_yaml_path}')
print(f'データセットパス: {DATASET_DIR} (RGB)')
print()
print(data_yaml)

---
## Step 3: カスタムモデルYAML作成

YOLOv8n (width=0.25) からチャネル数を大幅に削減した
超軽量モデルを定義する。

### モデル構成 (Issue #126: 方針A)

| 項目 | 値 | 備考 |
|------|-----|------|
| **width_multiple** | 0.08 | 動作実績あり (366KB INT8) |
| **max_channels** | 256 | 深い層のチャネル数を制限 |
| **入力チャネル** | 3 (RGB) | MCU前処理に合わせる |
| **精度改善** | #101パラメータ | epochs=300, cos_lr, lrf=0.001 |

In [ ]:
#############################################
# モデル設定 (Issue #126: 方針A)
# pico: width=0.08, max_channels=256 (動作実績あり)
# RGB 3ch + デフォルト学習パラメータ
#
# 注意: cos_lr=True + lrf=0.001 ではNPU上で
# 重みが飽和するため、デフォルト値を使用する。
#############################################
MODEL_VARIANT = 'pico'

# 入力設定
IMG_SIZE = 192
INPUT_CHANNELS = 3  # RGB (MCU前処理に合わせる)

# エポック数 (旧モデルと同じ)
EPOCHS = 200

# モデル設定 (動作実績構成: width=0.08, max_channels=256)
VARIANT_CONFIG = {
    'pico': {
        'depth_multiple': 0.33,
        'width_multiple': 0.08,
        'max_channels': 256,
    },
}

config = VARIANT_CONFIG[MODEL_VARIANT]
print(f'選択モデル: {MODEL_VARIANT}')
print(f'  depth_multiple: {config["depth_multiple"]}')
print(f'  width_multiple: {config["width_multiple"]}')
print(f'  max_channels: {config["max_channels"]}')
print(f'  入力: {IMG_SIZE}x{IMG_SIZE}x{INPUT_CHANNELS} (RGB)')
print(f'  エポック数: {EPOCHS}')

In [ ]:
# カスタムモデル YAML を動的に生成
model_yaml_content = f"""# YOLOv8-{MODEL_VARIANT}: Fall detection custom model
# depth_multiple={config['depth_multiple']}, width_multiple={config['width_multiple']}
# max_channels={config['max_channels']}
# Input: {IMG_SIZE}x{IMG_SIZE}x{INPUT_CHANNELS} (RGB)

nc: 1  # person only
ch: {INPUT_CHANNELS}  # RGB

scales:
  {MODEL_VARIANT}: [{config['depth_multiple']}, {config['width_multiple']}, {config['max_channels']}]

# YOLOv8 backbone
backbone:
  - [-1, 1, Conv, [64, 3, 2]]       # 0-P1/2
  - [-1, 1, Conv, [128, 3, 2]]      # 1-P2/4
  - [-1, 3, C2f, [128, True]]
  - [-1, 1, Conv, [256, 3, 2]]      # 3-P3/8
  - [-1, 6, C2f, [256, True]]
  - [-1, 1, Conv, [512, 3, 2]]      # 5-P4/16
  - [-1, 6, C2f, [512, True]]
  - [-1, 1, Conv, [1024, 3, 2]]     # 7-P5/32
  - [-1, 3, C2f, [1024, True]]
  - [-1, 1, SPPF, [1024, 5]]        # 9

# YOLOv8 head
head:
  - [-1, 1, nn.Upsample, [None, 2, \"nearest\"]]
  - [[-1, 6], 1, Concat, [1]]       # cat backbone P4
  - [-1, 3, C2f, [512]]             # 12

  - [-1, 1, nn.Upsample, [None, 2, \"nearest\"]]
  - [[-1, 4], 1, Concat, [1]]       # cat backbone P3
  - [-1, 3, C2f, [256]]             # 15 (P3/8-small)

  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 12], 1, Concat, [1]]      # cat head P4
  - [-1, 3, C2f, [512]]             # 18 (P4/16-medium)

  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 9], 1, Concat, [1]]       # cat head P5
  - [-1, 3, C2f, [1024]]            # 21 (P5/32-large)

  - [[15, 18, 21], 1, Detect, [nc]]  # Detect(P3, P4, P5)
"""

model_yaml_path = os.path.join(WORK_DIR, f'yolov8-{MODEL_VARIANT}-fall.yaml')
with open(model_yaml_path, 'w') as f:
    f.write(model_yaml_content)

print(f'モデルYAML作成完了: {model_yaml_path}')
print()
print(model_yaml_content)

In [ ]:
# モデル構造とパラメータ数を事前確認
from ultralytics import YOLO

# カスタムYAMLからモデル構築 (重みなし、アーキテクチャのみ)
model_check = YOLO(model_yaml_path)
print(f'\n=== {MODEL_VARIANT} モデル情報 ===')
print(model_check.info())

# パラメータ数の確認
total_params = sum(p.numel() for p in model_check.model.parameters())
print(f'\n総パラメータ数: {total_params:,} ({total_params/1e6:.3f}M)')
print(f'推定INT8サイズ: {total_params/1024:.0f} KB')

# 参考値との比較
print(f'\n--- 参考 ---')
print(f'YOLO-Fastest (顔認識): 0.24M params, 412KB INT8')
print(f'YOLOv8n (現在):        3.0M params, 3,149KB INT8')
print(f'Arena上限:             432KB')

---
## Step 4: モデル学習

カスタムYAMLからスクラッチで学習する (事前学習重みなし)。

### 学習パラメータ (デフォルト値、NPU互換)
- エポック数 200
- 線形学習率スケジュール (cos_lr=False)
- 最終学習率比 0.01 (lrf=0.01)
- patience=50, close_mosaic=10 (デフォルト)

**NPU非互換パラメータ（使用禁止）:**
- ~~cos_lr=True~~: 重みが極端な値になりNPUで飽和する
- ~~lrf=0.001~~: 学習終盤の学習率が低すぎてNPU量子化と相性が悪い

### 再学習時の注意

前回の学習結果がGoogle Driveに残っていると自動的に再開してしまいます。
新規学習をやり直す場合は、Step 4のセルを実行する前に以下を実行してください:

```python
!rm -rf /content/drive/MyDrive/yolo_training_pico/train
```

### 接続切れからの再開

学習結果はGoogle Drive上に自動保存されます。
接続が切れた場合:
1. ランタイムを再起動
2. Step 1 から Step 3 まで順に再実行
3. Step 4 のセルを実行すると、`last.pt` を検出して自動的に学習を再開します

In [ ]:
import os
from ultralytics import YOLO

# Google Drive上に学習出力先を設定 (接続切れ対策)
GDRIVE_TRAIN_DIR = '/content/drive/MyDrive/yolo_training_pico'
os.makedirs(GDRIVE_TRAIN_DIR, exist_ok=True)

# 前回の学習が中断された場合、last.pt から再開
last_pt = os.path.join(GDRIVE_TRAIN_DIR, 'train', 'weights', 'last.pt')

if os.path.exists(last_pt):
    print('=== 前回の学習を再開 ===')
    print(f'再開ポイント: {last_pt}')
    model = YOLO(last_pt)
    results = model.train(resume=True)
else:
    print('=== 新規学習開始 ===')
    # カスタムYAMLからモデル構築 (スクラッチ学習)
    model = YOLO(model_yaml_path)

    print(f'モデル: {MODEL_VARIANT}, 入力: {IMG_SIZE}x{IMG_SIZE}x{INPUT_CHANNELS}')

    results = model.train(
        data=data_yaml_path,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=64,
        device=0,
        workers=2,
        project=GDRIVE_TRAIN_DIR,
        name='train',
        exist_ok=True,
        # --- デフォルト学習パラメータ (NPU互換) ---
        optimizer='SGD',
        lr0=0.01,
        lrf=0.01,            # デフォルト値
        momentum=0.937,
        weight_decay=0.0005,
        warmup_epochs=3.0,
        cos_lr=False,        # デフォルト: 線形減衰
        # --- 早期終了・mosaic制御 (デフォルト) ---
        patience=50,
        close_mosaic=10,
        # --- データ拡張 (RGB向け: 色augmentation有効) ---
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,
        degrees=0.0,
        translate=0.1,
        scale=0.5,
        fliplr=0.5,
        mosaic=1.0,
    )

print('\n=== 学習完了 ===')

In [ ]:
# 学習曲線の表示
from IPython.display import Image, display
import os

results_png = os.path.join(GDRIVE_TRAIN_DIR, 'train', 'results.png')
if os.path.exists(results_png):
    display(Image(filename=results_png, width=800))
else:
    print('学習結果の画像が見つかりません')

---
## Step 5: 精度評価 (mAP)

In [ ]:
# best.pt で検証データセットを評価
best_pt = os.path.join(GDRIVE_TRAIN_DIR, 'train', 'weights', 'best.pt')
model = YOLO(best_pt)

metrics = model.val(
    data=data_yaml_path,
    imgsz=IMG_SIZE,
    batch=64,
    device=0,
    split='val',
)

print(f'\n=== 検証データ評価結果 ({MODEL_VARIANT}) ===')
print(f'mAP@0.5     : {metrics.box.map50:.4f} ({metrics.box.map50*100:.1f}%)')
print(f'mAP@0.5:0.95: {metrics.box.map:.4f} ({metrics.box.map*100:.1f}%)')
print(f'Precision    : {metrics.box.mp:.4f}')
print(f'Recall       : {metrics.box.mr:.4f}')

# 比較表
print(f'\n--- 他モデルとの比較 ---')
print(f'{"モデル":>30s}  mAP50  Recall')
print('-' * 55)
print(f'{"YOLOv8n RGB 100ep":>30s}  67.8%  59.3%')
print(f'{"pico RGB 200ep (旧版)":>30s}  45.9%  44.1%')
print(f'{"pico RGB 300ep (#126)":>30s}  {metrics.box.map50*100:.1f}%  {metrics.box.mr*100:.1f}%')

# KPI 判定
print(f'\n--- KPI 判定 ---')
map_pass = metrics.box.map50 >= 0.9
recall_pass = metrics.box.mr >= 0.8
print(f'mAP@0.5 >= 90%  : {"PASS" if map_pass else "FAIL"} ({metrics.box.map50*100:.1f}%)')
print(f'Recall >= 80%   : {"PASS" if recall_pass else "FAIL"} ({metrics.box.mr*100:.1f}%)')

# 旧版との比較
if metrics.box.map50 > 0.459:
    improvement = (metrics.box.map50 - 0.459) * 100
    print(f'\n>>> 旧版 pico (mAP=45.9%) から +{improvement:.1f}pt 改善')
else:
    degradation = (0.459 - metrics.box.map50) * 100
    print(f'\n>>> 旧版 pico (mAP=45.9%) から -{degradation:.1f}pt 劣化。パラメータ調整が必要')

In [ ]:
# テストデータセットでも評価
test_metrics = model.val(
    data=data_yaml_path,
    imgsz=IMG_SIZE,
    batch=64,
    device=0,
    split='test',
)

print(f'\n=== テストデータ評価結果 ({MODEL_VARIANT}) ===')
print(f'mAP@0.5     : {test_metrics.box.map50:.4f} ({test_metrics.box.map50*100:.1f}%)')
print(f'mAP@0.5:0.95: {test_metrics.box.map:.4f} ({test_metrics.box.map*100:.1f}%)')
print(f'Precision    : {test_metrics.box.mp:.4f}')
print(f'Recall       : {test_metrics.box.mr:.4f}')

---
## Step 6: ONNX エクスポート

In [ ]:
# ONNX エクスポート
model = YOLO(best_pt)

onnx_path = model.export(
    format='onnx',
    imgsz=IMG_SIZE,
    opset=11,
    simplify=True,
)

print(f'\nONNX エクスポート完了: {onnx_path}')
print(f'サイズ: {os.path.getsize(onnx_path)/1024:.1f} KB')

In [ ]:
import numpy as np
import glob
from PIL import Image

ONNX_PATH = onnx_path
SAVED_MODEL_DIR = os.path.join(WORK_DIR, 'saved_model')
FP32_PATH = os.path.join(WORK_DIR, f'model_{MODEL_VARIANT}_fp32.tflite')
INT8_PATH = os.path.join(WORK_DIR, f'model_{MODEL_VARIANT}_int8.tflite')

# Step 7a: ONNX -> SavedModel -> TFLite FP32
print('=== ONNX -> SavedModel (onnx2tf) ===')
!onnx2tf -i {ONNX_PATH} -o {SAVED_MODEL_DIR} -osd 2>&1 | tail -10

if os.path.isdir(SAVED_MODEL_DIR):
    import tensorflow as tf

    # FP32 TFLite
    print('\n=== FP32 TFLite 変換 ===')
    converter = tf.lite.TFLiteConverter.from_saved_model(SAVED_MODEL_DIR)
    tflite_fp32 = converter.convert()
    with open(FP32_PATH, 'wb') as f:
        f.write(tflite_fp32)
    print(f'FP32 TFLite: {os.path.getsize(FP32_PATH)/1024:.1f} KB')

    # Step 7b: INT8 量子化 (RGB)
    print('\n=== INT8 量子化 (RGB) ===')
    # RGB データセットからキャリブレーション画像を取得
    cal_dir = os.path.join(DATASET_DIR, 'images', 'val')
    cal_images = sorted(glob.glob(os.path.join(cal_dir, '*.jpg')))[:200]
    if not cal_images:
        cal_images = sorted(glob.glob(os.path.join(cal_dir, '*.png')))[:200]
    print(f'キャリブレーション画像: {len(cal_images)}枚 (RGB)')

    # FP32モデルの入力形状を取得
    interp = tf.lite.Interpreter(model_path=FP32_PATH)
    interp.allocate_tensors()
    inp_detail = interp.get_input_details()[0]
    inp_shape = inp_detail['shape']
    n, h, w, c = inp_shape
    print(f'入力形状: {inp_shape} (NHWC), ch={c}')

    def representative_dataset():
        for img_path in cal_images:
            img = Image.open(img_path).convert('RGB').resize((w, h))
            arr = np.array(img, dtype=np.float32) / 255.0
            arr = arr.reshape(1, h, w, 3)
            yield [arr]

    converter = tf.lite.TFLiteConverter.from_saved_model(SAVED_MODEL_DIR)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = representative_dataset
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8

    try:
        int8_model = converter.convert()
        with open(INT8_PATH, 'wb') as f:
            f.write(int8_model)
        int8_kb = os.path.getsize(INT8_PATH) / 1024
        print(f'INT8 TFLite: {int8_kb:.1f} KB')
    except Exception as e:
        print(f'INT8 量子化エラー: {e}')
        print('FP32 モデルは正常に生成されています。')
else:
    print('ERROR: SavedModel の生成に失敗しました')

In [ ]:
import tensorflow as tf
import numpy as np

ARENA_LIMIT_KB = 432

print(f'=== モデルサイズ検証 ({MODEL_VARIANT}) ===')
print()

# サイズ比較表
print('--- ファイルサイズ比較 ---')
for label, path in [('FP32', FP32_PATH), ('INT8', INT8_PATH)]:
    if os.path.exists(path):
        size_kb = os.path.getsize(path) / 1024
        status = 'OK' if size_kb <= ARENA_LIMIT_KB else 'OVER'
        print(f'{label}: {size_kb:.1f} KB ({size_kb/1024:.2f} MB) [{status}]')

# YOLOv8n との比較
if os.path.exists(INT8_PATH):
    int8_kb = os.path.getsize(INT8_PATH) / 1024
    reduction = (1 - int8_kb / 3149) * 100
    print(f'\n--- YOLOv8n INT8 (3,149KB) からの削減率: {reduction:.1f}% ---')
    print(f'\n--- Arena制約 ({ARENA_LIMIT_KB}KB) との比較 ---')
    if int8_kb <= ARENA_LIMIT_KB:
        margin = ARENA_LIMIT_KB - int8_kb
        print(f'OK: INT8モデル ({int8_kb:.0f}KB) <= Arena上限 ({ARENA_LIMIT_KB}KB) [余裕: {margin:.0f}KB]')
        print('注意: 実際のArenaサイズはMERA/Velaコンパイル結果で確認が必要です')
    else:
        over = int8_kb - ARENA_LIMIT_KB
        print(f'OVER: INT8モデル ({int8_kb:.0f}KB) > Arena上限 ({ARENA_LIMIT_KB}KB) [超過: {over:.0f}KB]')
        print('対策: width_multipleを更に縮小するか、入力サイズを160x160に縮小してください')

# INT8 モデル詳細
if os.path.exists(INT8_PATH):
    print(f'\n=== INT8 モデル詳細 ===')
    interp = tf.lite.Interpreter(model_path=INT8_PATH)
    interp.allocate_tensors()

    for tag, details in [('Input', interp.get_input_details()),
                         ('Output', interp.get_output_details())]:
        print(f'\n--- {tag} ---')
        for i, d in enumerate(details):
            print(f'  [{i}] {d["name"]} shape={d["shape"]} dtype={d["dtype"]}')
            qp = d.get('quantization_parameters', {})
            sc = qp.get('scales', np.array([]))
            zp = qp.get('zero_points', np.array([]))
            if len(sc) > 0:
                print(f'      scale={sc[0]:.8f}, zero_point={zp[0]}')

    # パラメータ数
    total_params = sum(p.numel() for p in YOLO(best_pt).model.parameters())
    print(f'\n--- パラメータ数: {total_params:,} ({total_params/1e6:.3f}M) ---')

---
## Step 9: 成果物ダウンロード

学習済みモデルを Google Drive に保存する。

In [ ]:
# Google Drive に成果物をコピー
import shutil

OUTPUT_DIR = f'/content/drive/MyDrive/fall_detection_model_{MODEL_VARIANT}'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 学習成果物のパス (GDRIVE_TRAIN_DIR ベース)
train_dir = os.path.join(GDRIVE_TRAIN_DIR, 'train')

# コピー対象ファイル
files_to_copy = {
    best_pt: f'best_{MODEL_VARIANT}.pt',
    os.path.join(train_dir, 'weights', 'last.pt'): f'last_{MODEL_VARIANT}.pt',
    os.path.join(train_dir, 'results.png'): 'results.png',
    os.path.join(train_dir, 'results.csv'): 'results.csv',
    ONNX_PATH: f'model_{MODEL_VARIANT}.onnx',
    FP32_PATH: f'model_{MODEL_VARIANT}_fp32.tflite',
    INT8_PATH: f'model_{MODEL_VARIANT}_int8.tflite',
    model_yaml_path: f'yolov8-{MODEL_VARIANT}-fall.yaml',
}

for src, dst_name in files_to_copy.items():
    if os.path.exists(src):
        dst = os.path.join(OUTPUT_DIR, dst_name)
        shutil.copy2(src, dst)
        size_kb = os.path.getsize(src) / 1024
        print(f'  {dst_name}: {size_kb:.1f} KB')

print(f'\n=== Google Drive に保存完了 ===')
print(f'場所: {OUTPUT_DIR}')

In [ ]:
# INT8 TFLite モデルを直接ダウンロード
from google.colab import files

if os.path.exists(INT8_PATH):
    files.download(INT8_PATH)
    print('INT8モデルのダウンロードを開始しました')
else:
    print('INT8モデルが見つかりません。Step 7 を先に実行してください。')

---
## まとめ

### 生成される成果物

| ファイル | 説明 |
|---|---|
| `best_{variant}.pt` | 学習済み PyTorch モデル (最良 mAP) |
| `model_{variant}.onnx` | ONNX 形式モデル |
| `model_{variant}_fp32.tflite` | TFLite FP32 モデル |
| `model_{variant}_int8.tflite` | TFLite INT8 量子化モデル |
| `yolov8-{variant}-fall.yaml` | カスタムモデル設定 |
| `results.png` | 学習曲線チャート |
| `results.csv` | 学習ログ (CSV) |

### 次のステップ (Issue #126)

1. Colab で学習を実行し、INT8 モデルを取得
2. `dataset/models/yolov8_pico_fall_int8.tflite` を新しいモデルで置き換え
3. `scripts/deploy_fall_detection.ps1` で MERA 変換
4. e2studio ビルド・実機書き込み
5. `ai detect` で人物検出が動作することを確認
6. `fall status` で転倒判定が正しく動作することを確認

### 備考

- 入力チャネルは 3ch (RGB) で学習しています (MCU前処理に一致)
- スクラッチ学習のため、COCO事前学習済み重みは使用していません
- 実際のArenaサイズはMERA変換結果で確定します
- モデル構成: pico (width=0.08, max_channels=256) は動作実績あり